## __Imports and Data Load__

In [3]:
# import
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, train_test_split, GridSearchCV, RandomizedSearchCV

In [4]:
pd.set_option('display.max_columns', None)

In [6]:
# load data
df = pd.read_parquet('../data/processed/01_df_combined.parquet', engine='pyarrow')

df.head()

,store_id,item_id,date,category_name,sold_quantity,price,revenue,zipcode,holiday_name,holiday_type,wind_dir,weather_code,temperature,wind_speed,wind_degree,precip,humidity,visibility,pressure,cloudcover
0,12,8,2025-01-01,Brot,4.0,7.30,29.200001,52064,Neujahr,holiday,SW,116,6.5,40.900002,223.399994,0.1,67.699997,9.8,1014.099976,55.500000
1,12,8,2025-01-02,Brot,22.0,7.30,160.600006,52064,Weihnachtsferien,school_holiday,NW,317,2.5,13.900000,280.799988,0.7,89.099998,8.0,1014.200012,82.300003
2,12,8,2025-01-03,Brot,19.0,7.30,138.699997,52064,Weihnachtsferien,school_holiday,WSW,116,0.3,18.600000,251.500000,0.0,87.400002,9.2,1023.099976,59.400002
3,12,8,2025-01-04,Brot,41.0,7.26,297.660004,52064,Weihnachtsferien,school_holiday,SSE,122,-0.2,9.200000,166.800003,0.0,88.900002,7.8,1019.400024,82.800003
4,12,8,2025-01-05,Brot,8.0,7.30,58.400002,52064,Weihnachtsferien,school_holiday,SW,266,6.0,26.299999,184.800003,0.8,89.699997,6.1,997.700012,88.699997


In [7]:
# missing values
missings = df.isna().sum()
print('Columns with missing values:')
print(missings[missings > 0])
print()
print('''
Observation/Explanation ☝:
In the last step (notebook) we have already created a consecutive time range for creating laggin and rolling features in this notebook.
This leads to a lot of missing values at dates where no items where sold.
After creating lags we wil delete all those NaN rows, because we will not need them anymore.
''')

Columns with missing values:
sold_quantity    31760525
price            31760525
revenue          31760525
dtype: int64


Observation/Explanation ☝:
In the last step (notebook) we have already created a consecutive time range for creating laggin and rolling features in this notebook.
This leads to a lot of missing values at dates where no items where sold.
After creating lags we wil delete all those NaN rows, because we will not need them anymore.



## __Feature Creation__

### __Lag Features__

In [8]:
pd.set_option('display.max_rows', 200)

In [9]:
# sorting first
df = df.sort_values(['store_id', 'item_id', 'date']).reset_index(drop=True)

In [10]:
# fill target NaN with 0
df['sold_quantity'] = df.sold_quantity.fillna(0)

In [11]:
# creating lags as dtpye float32 because of memory usage
lags = [1, 7, 14, 28]

for lag in lags:
    df[f'lag_{lag}'] = df.groupby(['store_id', 'item_id'])['sold_quantity'].shift(lag).astype('float32')
    

### __Rolling Features__

In [12]:
# creating rollings

rolling_means = [7, 14, 28]

# less writing, -> shift(1) to exclude the current target value of the calculation of the mean/sum/std
shifted_group = df.groupby(['store_id', 'item_id'])['sold_quantity'].shift(1)

#...mean and std
for r in rolling_means:
    df[f'rolling_mean_{r}'] = shifted_group.rolling(r).mean().astype('float32')
    df[f'rolling_std_{r}'] = shifted_group.rolling(r).std().astype('float32')

# ...sum
df['rolling_sum_7'] = shifted_group.rolling(7).sum().astype('float32')
df['rolling_sum_28'] = shifted_group.rolling(28).sum().astype('float32')

# ...expanding mean
df['expanding_mean'] = shifted_group.expanding().mean().astype('float32')


### __Calender Features__

### __Rounded Price Feature__

In [13]:
pd.set_option('display.max_rows', 300)

In [14]:
# first lets investigate price a little

# count of unique prices
print('Count of unique prices:', df.price.nunique())
print()

# example price range
print('Example price range from 2.20 - 2.50 Euro:')
display(pd.DataFrame(df.price.value_counts().sort_index())[220:250])

# observation
print('''
Observation:
There are very granular price changes. It seems that almost every price changes in 1-cent-steps.
To reduce complexity we round the price to 1 decimal (i.e 2.21, 2.22, 2.23,..., 2,29  -> 2.2))
''')

Count of unique prices: 1720

Example price range from 2.20 - 2.50 Euro:


,count
price,
2.21,1610
2.22,4240
2.23,2762
2.24,3566
2.25,28064
2.26,1410
2.27,1216
2.28,1839
2.29,1019



Observation:
There are very granular price changes. It seems that almost every price changes in 1-cent-steps.
To reduce complexity we round the price to 1 decimal (i.e 2.21, 2.22, 2.23,..., 2,29  -> 2.2))



In [15]:
# transform price to price_rounded
df['price_rounded'] = df.price.round(1)
df = df.drop('price', axis=1)

In [16]:
# check a sample
df[df.price_rounded > 0]['price_rounded'].sample(10)

28329690    0.7
4148149     9.1
822958      0.5
19768979    2.9
3068857     1.3
33181866    4.0
12836440    3.4
7845538     4.2
11623187    2.9
7915725     4.1
Name: price_rounded, dtype: float32

## __Removing all remaining missing values__

In [17]:
df_cleaned = df.dropna()

In [18]:
# check df_cleaned

print('df_cleaned final check:', end='\n\n')
print('Min date:', df_cleaned.date.min())
print('Max date:', df_cleaned.date.max())
print('Count of days:', df_cleaned.date.nunique())
print()
print('Missings values:', df_cleaned.isna().sum().sum())
print('Duplicates:', df_cleaned.duplicated(subset=['store_id', 'item_id', 'date']).sum())
print()
print(f'Available data for modeling: {df_cleaned.shape[0]:,} observations.')

df_cleaned final check:

Min date: 2025-01-29 00:00:00
Max date: 2026-04-30 00:00:00
Count of days: 455

Missings values: 0
Duplicates: 0

Available data for modeling: 3,449,803 observations.


## __Save df_cleaned to disc__

In [19]:
# save df_cleaned
df_cleaned.to_parquet('../data/processed/02_df_cleaned.parquet', index=False)